# 21 — ReflectiveAgent with Web Research

ReflectiveAgent that **searches the web** to verify facts, then reflects on quality, and revises until the threshold is met. The agent doesn't just write — it fact-checks itself.

**What you'll learn:**
1. ReflectiveAgent.with_builtins — searches web, writes, reflects, revises
2. Streaming reflections with quality scores
3. Custom reflection criteria
4. Memory across reflection runs

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shipit_agent import Agent, WebSearchTool
from shipit_agent.deep import ReflectiveAgent
from examples.run_multi_tool_agent import build_llm_from_env
from IPython.display import display, Markdown

llm = build_llm_from_env('bedrock')

def print_event(event):
    ICONS = {"run_started": "🚀", "run_completed": "✅", "step_started": "▶️",
             "tool_completed": "📦", "reasoning_started": "🧠", "reasoning_completed": "🧠"}
    icon = ICONS.get(event.type, "•")
    print(f"\n{icon} {event.message}")
    output = event.payload.get("output", "")
    if output and event.type in ("tool_completed", "run_completed"):
        print(f"{'─' * 60}")
        print(output[:600])
        if len(output) > 600: print(f"... ({len(output)} chars)")
        print(f"{'─' * 60}")
    quality = event.payload.get("quality")
    if quality is not None:
        bar = "█" * int(quality * 10) + "░" * (10 - int(quality * 10))
        print(f"  Quality: [{bar}] {quality:.2f}")
    feedback = event.payload.get("feedback")
    if feedback:
        print(f"  Feedback: {feedback[:200]}")

## 1. ReflectiveAgent.with_builtins — Web Research + Self-Improvement

The agent searches the web, writes content, reflects on quality, and revises. All with streaming.

In [ ]:
# ReflectiveAgent with ALL built-in tools — searches web, writes, reflects, revises
agent = ReflectiveAgent.with_builtins(
    llm=llm,
    reflection_prompt="Verify facts by searching the web. Check accuracy, completeness, and clarity. Score 0-1 honestly.",
    max_reflections=2,
    quality_threshold=0.8,
)

print("=== ReflectiveAgent — Web Research + Self-Improvement ===\n")
for event in agent.stream("Write a brief guide on deploying a Python app to AWS Lambda with Docker"):
    print_event(event)

## 2. With Specific Tools Only

In [ ]:
# Only web search — no code execution, no browser
agent = ReflectiveAgent(
    llm=llm,
    tools=[WebSearchTool()],
    reflection_prompt="Check if the content is factually accurate and well-structured. Score 0-1.",
    max_reflections=2,
    quality_threshold=0.85,
)

print("=== ReflectiveAgent — WebSearch Only ===\n")
result = agent.run("Explain the differences between REST and GraphQL APIs")

print(f"Final Quality: {result.final_quality:.2f}")
print(f"Revisions: {len(result.revisions)}")
for i, r in enumerate(result.reflections, 1):
    print(f"  Reflection {i}: quality={r.quality_score:.2f} — {r.feedback[:80]}...")
print(f"\n=== Final Output ===")
display(Markdown(result.output[:800]))